# Phase 1.5: Upgraded Face Geometry Reconstruction

**Objective:** Run identity-preserving 3D face geometry reconstruction with **anti-mean collapse layers**:
1. **Pixel3DMM Dense Fitting:** Surface normal & UV consistency to anchor geometry to photograph pixels.
2. **Stage 1.5 Macro-Shape Residual Network:** Non-linear vertex displacement predictor breaking FLAME linear PCA ceiling.
3. **Stage 5 Production Rig & Scaled Stylization:** 4 presets (neutral, chiseled, heroic, gigachad) with strict bitwise neck seam collar pinning.

In [ ]:
# Cell 1: Environment & Repository Setup
import os
import sys
import subprocess
from pathlib import Path

print("--- Setting Up Humanoid-Face-3D Workspace ---")
target_dir = Path("/kaggle/working/Humanoid-Face-3D")
if not (target_dir / "src/pipeline.py").exists():
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/NetPranav/Humanoid-Face-3D.git", str(target_dir)], check=True)
os.chdir(str(target_dir))
subprocess.run(["git", "fetch", "origin", "main"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=False)

if str(target_dir) not in sys.path:
    sys.path.insert(0, str(target_dir))
if "." not in sys.path:
    sys.path.insert(0, ".")
print("Working directory:", os.getcwd())


In [ ]:
# Cell 2: Install GPU Dependencies & Model Frameworks
import subprocess
subprocess.run(["pip", "install", "insightface", "onnxruntime-gpu", "trimesh", "pyyaml", "scipy", "Pillow", "opencv-python", "--quiet"], check=True)

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB VRAM)")


In [ ]:
# Cell 3: Discover & Mount Attached Datasets (FLAME & MICA)
import shutil
import subprocess
from pathlib import Path

print("--- Locating Model Weights in /kaggle/input ---")
flame_pkl_candidates = list(Path("/kaggle/input").glob("**/generic_model.pkl")) + list(Path(".").glob("**/generic_model.pkl"))
if not flame_pkl_candidates:
    raise FileNotFoundError("generic_model.pkl not found! Please attach dataset nightshowdown/flame-model.")
flame_pkl = flame_pkl_candidates[0]
print(f"  Found FLAME Model: {flame_pkl}")

Path("data/flame_model").mkdir(parents=True, exist_ok=True)
if not Path("data/flame_model/generic_model.pkl").exists():
    shutil.copy(flame_pkl, "data/flame_model/generic_model.pkl")

head_candidates = list(Path("/kaggle/input").glob("**/head_template.obj")) + list(Path(".").glob("**/head_template.obj"))
if head_candidates and not Path("data/flame_model/head_template.obj").exists():
    shutil.copy(head_candidates[0], "data/flame_model/head_template.obj")

# Locate MICA pretrained weights
Path("models_cache/mica").mkdir(parents=True, exist_ok=True)
dest_tar = Path("models_cache/mica/pretrained.tar")

if not dest_tar.exists() or dest_tar.stat().st_size < 100_000_000:
    mica_candidates = (
        list(Path("/kaggle/input").glob("**/pretrained.tar")) +
        list(Path("/kaggle/input").glob("**/mica.tar"))
    )
    if mica_candidates:
        print(f"  Mounting MICA weights from: {mica_candidates[0]}")
        shutil.copy(mica_candidates[0], dest_tar)
    else:
        print("  MICA dataset not attached. Streaming from Google Drive CDN (~5 seconds)...")
        subprocess.run([
            "curl", "-L", "--progress-bar",
            "https://drive.usercontent.google.com/download?id=1bYsI_spptzyuFmfLYqYkcJA6GZWZViNt&export=download&confirm=t",
            "-o", str(dest_tar)
        ], check=True)

if not dest_tar.exists() or dest_tar.stat().st_size < 100_000_000:
    raise RuntimeError("Failed to acquire MICA pretrained weights!")
print(f"All model assets verified: FLAME ({flame_pkl.name}) + MICA ({dest_tar.stat().st_size / 1e6:.1f} MB).")


In [ ]:
# Cell 4: Initialize Upgraded Pipeline (Pixel3DMM & Stage 1.5 Enabled)
import shutil
import subprocess
import yaml
from pathlib import Path
from src.pipeline import FaceGeoPipeline

# Create upgraded configuration enabling Stage 1.5 and dense fitting
config_data = {
    'pipeline': {'input_min_photos': 1},
    'stage0': {'min_face_confidence': 0.4},
    'stage1': {
        'fusion_space': 'embedding',
        'fusion_temperature': 0.05,
        'yaw_weight_exponent': 2,
        'enable_dense_fitting': True,
        'dense_iterations': 50
    },
    'stage1_5': {'enabled': True},
    'stage5': {
        'enable_lods': True,
        'enable_armature': True,
        'stylize': 'gigachad'
    }
}
upgraded_cfg_path = Path("configs/upgraded_run.yaml")
with open(upgraded_cfg_path, 'w') as f:
    yaml.dump(config_data, f)

pipeline = FaceGeoPipeline(
    config_path=str(upgraded_cfg_path),
    model_dir="models_cache"
)
print("Upgraded FaceGeoPipeline loaded successfully.")

# Prepare test subjects
test_dir = Path("data/test_subjects")
test_dir.mkdir(parents=True, exist_ok=True)

demo_inputs = list(Path("vendor/MICA/demo/input").glob("*.*"))
if demo_inputs:
    for img_p in demo_inputs:
        subj_name = img_p.stem
        s_folder = test_dir / subj_name
        s_folder.mkdir(parents=True, exist_ok=True)
        target = s_folder / img_p.name
        if not target.exists():
            shutil.copy(img_p, target)

if not any(test_dir.iterdir()):
    print("Fetching benchmark portraits from CDN fallback...")
    fallbacks = {
        "connelly": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/connelly.jpg",
        "carell": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/carell.jpg",
        "lawrence": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/lawrence.jpg"
    }
    for s_name, url in fallbacks.items():
        s_folder = test_dir / s_name
        s_folder.mkdir(parents=True, exist_ok=True)
        subprocess.run(["curl", "-L", "-s", url, "-o", str(s_folder / f"{s_name}.jpg")], check=True)

subject_folders = sorted([d for d in test_dir.iterdir() if d.is_dir() and any(d.glob("*.*"))])
print(f"Test subjects ready ({len(subject_folders)}): {[s.name for s in subject_folders]}")


In [ ]:
# Cell 5: Execute Upgraded Multi-Subject 3D Reconstruction
import json
import numpy as np
from pathlib import Path

output_base = Path("outputs/upgraded_inference")
output_base.mkdir(parents=True, exist_ok=True)

results = {}
subject_betas = {}

for s_folder in subject_folders:
    subj_name = s_folder.name
    photos = sorted([str(p) for p in s_folder.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    if not photos:
        continue

    print("\n==================================================")
    print(f"Reconstructing Subject: {subj_name} ({len(photos)} photo(s)) [Upgraded Pipeline]...")
    out_dir = output_base / subj_name
    res = pipeline.run(photos, str(out_dir))
    results[subj_name] = res

    with open(res["manifest_path"]) as f:
        manifest = json.load(f)
    subject_betas[subj_name] = np.array(manifest["beta_shape"], dtype=np.float32)
    print(f"  Neutral Mesh: {res['obj_path']}")
    print(f"  Blendshapes:  {res.get('blendshapes_path')}")
    print(f"  Armature Rig: {res.get('armature_path')}")


In [ ]:
# Cell 6: Full Stylization Preset Suite (Neutral, Chiseled, Heroic, Gigachad)
import trimesh
from src.stage5_export.exporter import Stage5Exporter

print("\n--- Generating Full Preset Suite with Preserved UE5 Armatures & LODs ---")
exporter = Stage5Exporter(enable_lods=True, enable_armature=True)
presets = ["chiseled", "heroic", "gigachad"]

for subj_name, res in results.items():
    s_out = output_base / subj_name
    mesh = trimesh.load(res["obj_path"], process=False)
    verts = mesh.vertices
    faces = mesh.faces

    for preset in presets:
        preset_dir = s_out / f"stylized_{preset}"
        m = exporter.export_production_asset(
            neutral_vertices=verts,
            faces=faces,
            output_dir=preset_dir,
            stylization_params=preset,
            export_fbx=False
        )
        max_d = (m.get("stylization") or {}).get("max_displacement", 0.0)
        print(f"  {subj_name} [{preset}]: max displacement = {max_d:.2f}mm | Saved to {preset_dir.name}/")


In [ ]:
# Cell 7: Phase 1 & 1.5 Gate Verification
import numpy as np
import trimesh

print("\n--- Running Phase 1.5 Upgraded Gate Verification ---")
# Gate 1: Non-collapse shape divergence
if len(subject_betas) >= 2:
    names = list(subject_betas.keys())
    gate_passed = True
    print("Pairwise Shape Divergence (||beta_a - beta_b||):")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            dist = float(np.linalg.norm(subject_betas[names[i]] - subject_betas[names[j]]))
            print(f"  {names[i]} vs {names[j]}: {dist:.4f}")
            if dist < 1e-3:
                print(f"  FAIL: {names[i]} and {names[j]} collapsed to identical shape!")
                gate_passed = False
    assert gate_passed, "Gate Failure: Identities collapsed to mean face!"
    print("GATE 1 PASSED: Identities are distinct and non-degenerate.")

# Gate 2: Neck Seam Pinning Contract (Delta v = 0 on lowest 20% collar vertices)
print("\nVerifying Critical Invariant 4 (Neck Seam Collar Pinning)...")
for subj_name, res in results.items():
    s_out = output_base / subj_name
    mesh_neutral = trimesh.load(res["obj_path"], process=False)
    y = mesh_neutral.vertices[:, 1]
    y_norm = (y - np.min(y)) / max(np.max(y) - np.min(y), 1e-4)
    collar_idx = np.where(y_norm <= 0.20)[0]

    for preset in ["chiseled", "heroic", "gigachad"]:
        stylized_obj = s_out / f"stylized_{preset}" / "head_mesh.obj"
        if stylized_obj.exists():
            mesh_stylized = trimesh.load(str(stylized_obj), process=False)
            deltas = mesh_stylized.vertices - mesh_neutral.vertices
            collar_deltas = deltas[collar_idx]
            max_collar_movement = float(np.max(np.abs(collar_deltas)))
            print(f"  {subj_name} [{preset}] neck collar max delta: {max_collar_movement:.6f} mm")
            assert max_collar_movement < 1e-5, f"Neck seam contract violated for {subj_name} [{preset}]!"

print("GATE 2 PASSED: Neck seam contract strictly verified (Delta v = 0 on collar).")

# Gate 3: Stylization Scale Magnitude Verification (0.08 - 0.12 * H range)
print("\nVerifying Gate 3 (Stylization Range Bounds)...")
for subj_name, res in results.items():
    s_out = output_base / subj_name
    mesh_neutral = trimesh.load(res["obj_path"], process=False)
    for preset, min_bound in [("chiseled", 10.0), ("gigachad", 18.0)]:
        stylized_obj = s_out / f"stylized_{preset}" / "head_mesh.obj"
        if stylized_obj.exists():
            mesh_stylized = trimesh.load(str(stylized_obj), process=False)
            deltas = mesh_stylized.vertices - mesh_neutral.vertices
            max_d = float(np.max(np.linalg.norm(deltas, axis=-1)))
            print(f"  {subj_name} [{preset}] max displacement: {max_d:.2f} mm")
            assert max_d >= min_bound, f"Stylization too weak ({max_d:.2f}mm < {min_bound}mm)!"

print("GATE 3 PASSED: Stylization achieves target artistic impact without distortion.")
print("\nAll Upgraded Pipeline Gates Successfully Cleared!")


In [ ]:
# Cell 8: Multi-Subject Visual Preview Grid & Asset Archive Creation
import cv2
import matplotlib.pyplot as plt
import subprocess
from pathlib import Path

n_subj = len(results)
if n_subj > 0:
    fig, axes = plt.subplots(1, min(4, n_subj), figsize=(4 * min(4, n_subj), 4))
    if n_subj == 1:
        axes = [axes]
    for ax, (subj_name, res) in zip(axes, list(results.items())[:4]):
        prev = res.get("preview_path")
        if prev and Path(prev).exists():
            img = cv2.imread(prev)[:, :, ::-1]
            ax.imshow(img)
            ax.set_title(f"{subj_name} (Upgraded 3D)")
            ax.axis("off")
        else:
            ax.text(0.5, 0.5, f"{subj_name}\nReconstruction OK", ha="center", va="center")
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(output_base / "upgraded_preview_grid.png", dpi=150)
    plt.show()
    print(f"Preview gallery saved to: {output_base / 'upgraded_preview_grid.png'}")

# Archive output for clean single-file download
archive_path = Path("/kaggle/working/upgraded_inference_assets.tar.gz")
print(f"\nArchiving output assets to {archive_path.name}...")
subprocess.run(["tar", "-czf", str(archive_path), "-C", "/kaggle/working/Humanoid-Face-3D/outputs", "upgraded_inference"], check=True)
print(f"Archive created successfully ({archive_path.stat().st_size / 1e6:.2f} MB).")
